# 🚀 AI BACKEND SERVER - GOOGLE COLAB (GPU TESLA T4)
### Dự án: Phat_trien_ung_dung_DT012
---
**Quy trình khởi động lần sau:**
1. Kéo thả file `serviceAccountKey.json` vào mục tệp bên trái (thư mục gốc `/content` hoặc `backend-ai`).
2. Chọn menu **Thời gian chạy (Runtime)** -> **Chạy tất cả (Run all)** (hoặc phím tắt `Ctrl + F9`).
3. Cuộn xuống ô cuối cùng để lấy link `https://xxxx.trycloudflare.com` dán vào App điện thoại.

In [ ]:
# ==============================================================================
# Ô 1: CÀI ĐẶT THƯ VIỆN HỆ THỐNG & CLONE MÃ NGUỒN
# ==============================================================================
!apt-get update -qq && apt-get install -y -qq ffmpeg zstd

# Xóa thư mục cũ nếu có để luôn clone bản mới nhất
!rm -rf /content/Phat_trien_ung_dung_DT012
!git clone -b Tuan8 https://github.com/Minhthang1472/Phat_trien_ung_dung_DT012.git

%cd /content/Phat_trien_ung_dung_DT012/backend-ai
!pip install -q -r requirements.txt

print("✅ HOÀN TẤT Ô 1: Đã clone mã nguồn và cài xong thư viện Python!")

In [ ]:
# ==============================================================================
# Ô 2: THIẾT LẬP CẤU HÌNH .ENV TỐI ƯU GPU CHO COLAB
# ==============================================================================
import shutil, os

# Nếu bạn đã upload file serviceAccountKey.json ở ngoài /content, tự động copy vào backend-ai
if os.path.exists('/content/serviceAccountKey.json'):
    shutil.copy('/content/serviceAccountKey.json', '/content/Phat_trien_ung_dung_DT012/backend-ai/serviceAccountKey.json')

with open('/content/Phat_trien_ung_dung_DT012/backend-ai/.env', 'w') as f:
    f.write("""WHISPER_DEVICE=cuda
WHISPER_COMPUTE_TYPE=float16
WHISPER_MODEL_SIZE=base
OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_MODEL=qwen2.5:3b
FIREBASE_CREDENTIALS_PATH=serviceAccountKey.json
""")

print("✅ HOÀN TẤT Ô 2: Cấu hình .env chuẩn GPU Tesla T4 đã tạo xong!")

In [ ]:
# ==============================================================================
# Ô 3: CÀI ĐẶT OLLAMA & TẢI MÔ HÌNH QWEN 2.5 LÊN GPU
# ==============================================================================
import subprocess, time

# Cài đặt Ollama Linux
!curl -fsSL https://ollama.com/install.sh | sh

# Khởi động Ollama Server chạy ngầm
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

# Nạp mô hình Qwen 2.5 (3B) vào VRAM GPU
!ollama pull qwen2.5:3b

print("\n✅ HOÀN TẤT Ô 3: Ollama & Qwen 2.5 3B đã sẵn sàng trên GPU!")

In [ ]:
# ==============================================================================
# Ô 4: BẬT CLOUDFLARE TUNNEL & KHỞI CHẠY SERVER FASTAPI
# ==============================================================================
import time, re

# Dọn dẹp tiến trình cũ
!pkill cloudflared
!pkill uvicorn

# Tải và chạy Cloudflare Tunnel chế độ nohup chống tràn bộ đệm
!wget -nc -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!nohup ./cloudflared tunnel --url http://127.0.0.1:8000 > tunnel.log 2>&1 &

print("⏳ Đang tạo đường hầm Cloudflare...")
time.sleep(8)

with open("tunnel.log", "r") as f:
    log_text = f.read()

urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_text)

if urls:
    print("\n" + "="*60)
    print(f"🎉 ĐƯỜNG LINK BACKEND CỦA BẠN LÀ:\n👉 {urls[-1]}")
    print("="*60)
    print(f"Kiểm tra Swagger Docs: {urls[-1]}/docs\n")
else:
    print("Chưa lấy được link tự động, vui lòng xem tunnel.log:")
    print(log_text)

# Khởi chạy FastAPI Server
%cd /content/Phat_trien_ung_dung_DT012/backend-ai
!uvicorn app.main:app --host 0.0.0.0 --port 8000